# PP-OCRv6 + VLM Correction Pipeline — Colab Runner

Notebook này hướng dẫn chạy pipeline OCR + VLM correction **từng bước** trên Google Colab.

**Yêu cầu:** Chọn Runtime → Change runtime type → **GPU (T4)** trước khi bắt đầu.

---

## Tổng quan các bước

| Bước | Script | Mô tả | GPU? |
|------|--------|-------|------|
| 1 | `01_build_frame_registry.py` | Quét frames, tạo metadata | ❌ |
| 2 | `02_run_ppocr.py` | PP-OCRv6 detect + recognize | ✅ |
| 3 | `03_build_vlm_jobs.py` | Risk scoring + grouping + crop | ❌ |
| 4 | `04_run_vlm_correction.py` | VLM sửa lỗi chính tả | ✅ |
| 5 | `05_merge_features.py` | Gộp raw + corrected OCR | ❌ |
| 6 | `06_build_es_documents.py` | Tạo JSONL cho Elasticsearch | ❌ |
| 6.5 | `run_vlm_benchmark.py` | Benchmark chọn model VLM | ✅ |

> ⚠️ **Lưu ý xung đột CUDA:** PaddleOCR và VLM (transformers + bitsandbytes) có thể xung đột thư viện CUDA. Nếu gặp lỗi, hãy **Restart Runtime** giữa Bước 2 và Bước 4.

---
## 0. Kiểm tra GPU & Clone code

In [ ]:
# Kiểm tra GPU khả dụng
!nvidia-smi

In [ ]:
# Clone code từ GitHub repo AIC-Test, nhánh Khoa
import os
if os.path.exists('/content/AIC-Test/.git'):
    os.chdir('/content/AIC-Test')
    !git fetch origin Khoa
    !git checkout Khoa
    !git pull --ff-only origin Khoa
else:
    !git clone -b Khoa https://github.com/KwanFam26022005/AIC-Test.git /content/AIC-Test

# Đường dẫn gốc tới pipeline
PIPELINE_DIR = "/content/AIC-Test/data_processing/OCR/ocr_vlm_pipeline"
print(f"Pipeline directory: {PIPELINE_DIR}")

In [ ]:
# Cài đặt các thư viện cơ bản (không GPU-specific)
!pip install -q pandas pyarrow Pillow PyYAML tqdm matplotlib seaborn

---
## 0.1. Chuẩn bị dữ liệu frames

Bạn cần upload hoặc mount thư mục chứa frames `.jpg` của video cần xử lý.

**Tùy chọn A:** Upload thủ công lên Colab (phù hợp demo nhỏ ~vài trăm frames).

**Tùy chọn B:** Mount Google Drive chứa dữ liệu (phù hợp scale lớn).

In [ ]:
# === TÙY CHỌN A: Sử dụng dữ liệu mẫu nhỏ để test ===
# Tạo thư mục frames mẫu (thay bằng ảnh thật của bạn)
import os
FRAMES_DIR = "/content/frames/L25_V001"
os.makedirs(FRAMES_DIR, exist_ok=True)

# Nếu bạn đã có frames, upload vào thư mục trên
# hoặc copy từ Google Drive
print(f"Frames directory: {FRAMES_DIR}")
print(f"Số frames hiện có: {len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])}")

In [ ]:
# === TÙY CHỌN B: Mount Google Drive ===
# Bỏ comment các dòng dưới nếu frames nằm trong Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# FRAMES_DIR = "/content/drive/MyDrive/AIC2026/frames/L25_V001"
# print(f"Frames directory: {FRAMES_DIR}")
# print(f"Số frames: {len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])}")

In [ ]:
# Tạo config Colab runtime (ghi đè đường dẫn phù hợp)
import yaml, os

VIDEO_ID = "L25_V001"  # <-- thay đổi nếu cần
OUTPUT_DIR = f"/content/outputs/{VIDEO_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Smoke test: đặt 10 để chạy nhanh. Full run: đặt None.
VLM_LIMIT = 10

# Benchmark nhỏ trên Colab trước, tăng lên khi đã ổn định env/model.
BENCHMARK_SAMPLE_GROUPS = 10
BENCHMARK_MODELS = [
    "5CD-AI/Vintern-3B-beta",
    "5CD-AI/Vintern-3B-R-beta",
    "erax-ai/EraX-VL-2B-V1.5",
]

colab_config = {
    "extends": "default.yaml",
    "project": {
        "video_id": VIDEO_ID,
        "frames_dir": FRAMES_DIR,
        "output_dir": OUTPUT_DIR,
        "resume": True,
    },
    "runtime": {
        "device": "cuda:0",
        "num_workers_io": 2,
    },
    "ppocr": {
        # PaddleX imports ModelScope at import-time; this avoids torch/NCCL crashes in Colab.
        "model_source": "huggingface",
        "stub_modelscope": True,
    },
    "vlm": {
        "cache_path": f"{OUTPUT_DIR}/vlm_cache.sqlite",
        # Colab T4/Python 3.12 often cannot build flash-attn. Use eager attention.
        "attn_implementation": "eager",
        "stub_flash_attn": True,
        # Some accelerate/transformers paths still call .to() on 4-bit models.
        "patch_quantized_to": True,
        # Keep the quantized VLM on one GPU to avoid Qwen2 CPU/CUDA rotary mismatch.
        "device_map": "cuda:0",
        "patch_qwen2_rotary_device": True,
    },
}

config_path = f"{PIPELINE_DIR}/configs/colab_runtime.yaml"
with open(config_path, "w") as f:
    yaml.dump(colab_config, f, default_flow_style=False)

print(f"Config saved to: {config_path}")
!cat {config_path}

---
## Bước 1: Tạo Frame Registry

Quét thư mục frames, đọc metadata (kích thước, dung lượng file), tạo danh sách tất cả frame.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/01_build_frame_registry.py --config configs/colab_runtime.yaml

In [ ]:
# === QA Bước 1: Frame Registry ===
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

registry = pd.read_parquet(f"{OUTPUT_DIR}/frame_registry.parquet")
print(f"Tổng số frames: {len(registry):,}")
print(f"Video ID: {registry['video_id'].iloc[0] if len(registry) else VIDEO_ID}")
print(f"Trạng thái source_status: {registry['source_status'].value_counts(dropna=False).to_dict()}")

if len(registry):
    print(f"Frame number range: {registry['frame_number'].min()} -> {registry['frame_number'].max()}")
    print(f"Kích thước phổ biến:")
    display(registry.groupby(['width', 'height']).size().reset_index(name='count').sort_values('count', ascending=False).head(10))
    display(registry.head(10))

    sample = registry[registry['source_status'].eq('ok')].head(8)
    if len(sample):
        fig, axes = plt.subplots(2, 4, figsize=(14, 7))
        for ax, (_, row) in zip(axes.flat, sample.iterrows()):
            img = Image.open(row['frame_path'])
            ax.imshow(img)
            ax.set_title(f"{row['frame_id']} | {row['width']}x{row['height']}", fontsize=9)
            ax.axis('off')
        for ax in axes.flat[len(sample):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("Không có frame nào. Kiểm tra lại FRAMES_DIR trước khi chạy các bước sau.")

---
## Bước 2: Chạy PP-OCRv6

Sử dụng PP-OCRv6 để detect và recognize text trên tất cả frames.

> ⚠️ Bước này cần cài đặt PaddleOCR. Nếu bạn đã chạy xong bước này, có thể **Restart Runtime** rồi nhảy sang Bước 3.

In [ ]:
# Cài đặt PaddlePaddle GPU + PaddleOCR
# PP-OCRv6 cần PaddlePaddle mới hơn 3.0.0; bản 3.0.0 có thể lỗi PIR `strides` trên Colab.
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu
!pip install -q --upgrade paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q --upgrade "paddleocr>=3.0.0"

In [ ]:
# Kiểm tra PaddlePaddle đã cài đặt đúng GPU
import paddle
print(f"PaddlePaddle version: {paddle.__version__}")
print(f"CUDA available: {paddle.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")
major, minor, *_ = [int(x) for x in paddle.__version__.split('.')[:2]]
assert (major, minor) >= (3, 3), "PP-OCRv6 cần PaddlePaddle >= 3.3.0. Hãy restart runtime và chạy lại cell cài đặt Paddle."

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/02_run_ppocr.py --config configs/colab_runtime.yaml

In [ ]:
# === QA Bước 2: PP-OCR Raw ===
import pandas as pd
import matplotlib.pyplot as plt

registry = pd.read_parquet(f"{OUTPUT_DIR}/frame_registry.parquet")
raw = pd.read_parquet(f"{OUTPUT_DIR}/ppocr_raw.parquet")
print(f"Tổng số OCR lines: {len(raw):,}")
print(f"Số frames có text: {raw['frame_id'].nunique():,}")
print(f"Tỷ lệ frames có text: {raw['frame_id'].nunique()/max(1, len(registry))*100:.1f}%")
print(f"Confidence mean/min/p50/p95: {raw['confidence'].mean():.3f} / {raw['confidence'].min():.3f} / {raw['confidence'].median():.3f} / {raw['confidence'].quantile(0.95):.3f}")
print(f"Lines confidence < 0.75: {(raw['confidence'] < 0.75).sum():,} ({(raw['confidence'] < 0.75).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
raw['confidence'].hist(bins=30, ax=axes[0])
axes[0].set_title('Confidence distribution')
axes[0].set_xlabel('confidence')
axes[0].set_ylabel('num lines')

lines_per_frame = raw.groupby('frame_id').size()
lines_per_frame.hist(bins=30, ax=axes[1])
axes[1].set_title('OCR lines per frame')
axes[1].set_xlabel('num lines')
axes[1].set_ylabel('num frames')
plt.tight_layout()
plt.show()

print("--- Low-confidence sample ---")
display(raw.nsmallest(15, 'confidence')[['frame_id', 'line_idx', 'ocr_text', 'confidence', 'bbox']])
print("--- OCR sample ---")
display(raw[['frame_id', 'line_idx', 'ocr_text', 'confidence', 'bbox']].head(15))

---
## Bước 3: Đánh giá rủi ro, Phân cụm & Tạo VLM Jobs

- Tính `risk_score` cho từng dòng OCR.
- Gom các bounding box gần nhau thành group.
- Cắt ảnh crop cho các group cần sửa lỗi.
- Lọc ra các VLM jobs (chỉ gửi group có rủi ro cao cho VLM).

> 💡 Bước này KHÔNG cần GPU, chạy thuần CPU.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/03_build_vlm_jobs.py --config configs/colab_runtime.yaml

In [ ]:
# === QA Bước 3a: Risk Scoring ===
import pandas as pd
import matplotlib.pyplot as plt

risk = pd.read_parquet(f"{OUTPUT_DIR}/ocr_risk.parquet")
threshold = 0.45
print(f"Tổng dòng OCR: {len(risk):,}")
print(f"need_vlm_line=True: {risk['need_vlm_line'].sum():,} ({risk['need_vlm_line'].mean()*100:.1f}%)")
print(f"risk_score mean/p50/p95/max: {risk['risk_score'].mean():.3f} / {risk['risk_score'].median():.3f} / {risk['risk_score'].quantile(0.95):.3f} / {risk['risk_score'].max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
risk['risk_score'].hist(bins=25, ax=axes[0])
axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold}')
axes[0].set_title('Risk score distribution')
axes[0].legend()

reasons = risk[['risk_reasons']].explode('risk_reasons').dropna()
if len(reasons):
    reasons['risk_reasons'].value_counts().plot(kind='bar', ax=axes[1])
    axes[1].set_title('Risk reasons')
    axes[1].tick_params(axis='x', rotation=30)
else:
    axes[1].text(0.5, 0.5, 'No risk reasons', ha='center')
    axes[1].axis('off')
plt.tight_layout()
plt.show()

print("--- Các dòng risk cao nhất ---")
display(risk.nlargest(20, 'risk_score')[['frame_id', 'line_idx', 'ocr_text', 'confidence', 'risk_score', 'risk_reasons', 'need_vlm_line']])

print("--- High confidence nhưng vẫn bị flag, thường là lỗi dấu/mojibake ---")
display(risk[(risk['confidence'] >= 0.90) & (risk['need_vlm_line'])][['frame_id', 'line_idx', 'ocr_text', 'confidence', 'risk_score', 'risk_reasons']].head(20))

In [ ]:
# === QA Bước 3b: Grouping & VLM Jobs ===
groups = pd.read_parquet(f"{OUTPUT_DIR}/ocr_groups.parquet")
print(f"Tổng số groups: {len(groups):,}")
print(f"Groups cần VLM: {groups['need_vlm_group'].sum():,} ({groups['need_vlm_group'].mean()*100:.1f}%)")
print(f"Avg lines/group: {groups['line_indices'].map(len).mean():.2f}")

display(groups['region_type'].value_counts().rename_axis('region_type').reset_index(name='count'))

jobs = groups[groups['need_vlm_group']].copy()
jobs['has_crop'] = jobs['crop_path'].notna() & jobs['crop_path'].map(lambda p: os.path.exists(p) if isinstance(p, str) else False)
print(f"VLM jobs: {len(jobs):,}")
print(f"Jobs có crop hợp lệ: {jobs['has_crop'].sum():,}/{len(jobs):,}")
if 'crop_error' in jobs.columns:
    print(f"Crop errors: {jobs['crop_error'].dropna().value_counts().to_dict()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
groups['max_risk_score'].hist(bins=25, ax=axes[0])
axes[0].set_title('Group max risk score')
groups['line_indices'].map(len).hist(bins=20, ax=axes[1])
axes[1].set_title('Lines per group')
plt.tight_layout()
plt.show()

print("--- Top VLM jobs by max_risk_score ---")
display(jobs.sort_values('max_risk_score', ascending=False)[['frame_id', 'group_id', 'region_type', 'line_indices', 'raw_group_text', 'min_confidence', 'avg_confidence', 'max_risk_score', 'crop_path']].head(20))

bad_jobs = jobs[~jobs['has_crop']]
if len(bad_jobs):
    print("CẢNH BÁO: Có VLM jobs thiếu crop_path hợp lệ. Nên sửa crop trước khi chạy Bước 4.")
    display(bad_jobs[['frame_id', 'group_id', 'raw_group_text', 'crop_path']].head(20))

crops_dir = f"{OUTPUT_DIR}/crops"
if os.path.exists(crops_dir):
    crop_files = [f for f in os.listdir(crops_dir) if f.endswith('.jpg')]
    print(f"Số ảnh crop đã tạo: {len(crop_files):,}")

In [ ]:
# === Xem minh họa ảnh crop mẫu ===
import matplotlib.pyplot as plt
from PIL import Image
import os

crops_dir = f"{OUTPUT_DIR}/crops"
if os.path.exists(crops_dir) and 'jobs' in globals():
    crop_rows = jobs[jobs['has_crop']].sort_values('max_risk_score', ascending=False).head(8)
    if len(crop_rows):
        fig, axes = plt.subplots(2, 4, figsize=(16, 6))
        for idx, (ax, (_, row)) in enumerate(zip(axes.flat, crop_rows.iterrows())):
            img = Image.open(row['crop_path'])
            ax.imshow(img)
            ax.set_title(f"{row['frame_id']} g{row['group_id']} | risk={row['max_risk_score']:.2f}", fontsize=8)
            ax.axis('off')
        for ax in axes.flat[len(crop_rows):]:
            ax.axis('off')
        plt.suptitle('Top risk crops sẽ gửi cho VLM', fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("Không có ảnh crop nào (có thể tất cả OCR lines đều confidence cao).")
else:
    print(f"Thư mục crops chưa tồn tại: {crops_dir}")

In [ ]:
# === Xem VLM Jobs ===
import os
vlm_jobs_path = f"{OUTPUT_DIR}/vlm_jobs.parquet"
if os.path.exists(vlm_jobs_path):
    jobs = pd.read_parquet(vlm_jobs_path)
    print(f"Số VLM jobs cần xử lý: {len(jobs)}")
    jobs[['frame_id', 'group_id', 'region_type', 'raw_group_text', 'max_risk_score']].head(10)
else:
    print("Chưa có vlm_jobs.parquet")

---
## ⚠️ QUAN TRỌNG: Restart Runtime nếu cần

Nếu bạn đã cài PaddlePaddle ở Bước 2 và bây giờ cần cài VLM stack (transformers + bitsandbytes), hãy:

1. **Runtime → Restart session** (hoặc Ctrl+M → .)
2. Sau khi restart, chạy lại cell **Clone code** (mục 0) và cell **Config** để khôi phục biến.
3. Sau đó tiếp tục từ Bước 4 bên dưới.

Nếu không gặp lỗi xung đột, bạn có thể bỏ qua bước restart và chạy tiếp.

In [ ]:
# === Chạy cell này SAU KHI restart runtime ===
# Khôi phục lại các biến đường dẫn

PIPELINE_DIR = "/content/AIC-Test/data_processing/OCR/ocr_vlm_pipeline"
VIDEO_ID = "L25_V001"  # <-- giữ đúng với giá trị đã dùng ở trên
FRAMES_DIR = "/content/frames/L25_V001"  # <-- giữ đúng với giá trị đã dùng ở trên
OUTPUT_DIR = f"/content/outputs/{VIDEO_ID}"
VLM_LIMIT = 10  # Full run: đặt None
BENCHMARK_SAMPLE_GROUPS = 10
BENCHMARK_MODELS = [
    "5CD-AI/Vintern-3B-beta",
    "5CD-AI/Vintern-3B-R-beta",
    "erax-ai/EraX-VL-2B-V1.5",
]

print(f"Pipeline: {PIPELINE_DIR}")
print(f"Frames:   {FRAMES_DIR}")
print(f"Output:   {OUTPUT_DIR}")

import os
for f in ['frame_registry.parquet', 'ppocr_raw.parquet', 'ocr_risk.parquet', 'ocr_groups.parquet', 'vlm_jobs.parquet']:
    path = os.path.join(OUTPUT_DIR, f)
    status = '✅' if os.path.exists(path) else '❌'
    print(f"  {status} {f}")

---
## Bước 4: Chạy VLM Correction

Cài đặt stack VLM (transformers, bitsandbytes) và chạy sửa lỗi chính tả tiếng Việt bằng Vintern-3B.

> ✅ Bước này cần GPU. Model Vintern-3B ở chế độ 4-bit chỉ dùng ~3.5-5GB VRAM, phù hợp T4 (16GB).

In [ ]:
# Cài đặt VLM dependencies
!pip install -q transformers==4.44.2 accelerate einops timm sentencepiece bitsandbytes torchvision
!pip install -q pandas pyarrow Pillow PyYAML tqdm

In [ ]:
# Kiểm tra GPU + torch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_vram = getattr(props, 'total_memory', getattr(props, 'total_mem', None))
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    if total_vram is not None:
        print(f"VRAM: {total_vram / 1024**3:.1f} GB")
    else:
        print(f"VRAM: unknown; available fields: {[name for name in dir(props) if not name.startswith('_')]}")

In [ ]:
# Chạy VLM correction
# VLM_LIMIT=10 để smoke test nhanh; VLM_LIMIT=None để chạy full jobs.
import os
import subprocess
import time
from datetime import datetime

%cd {PIPELINE_DIR}
limit_arg = "" if VLM_LIMIT is None else f"--limit {VLM_LIMIT}"
jobs_path = f"{OUTPUT_DIR}/vlm_jobs.parquet"
if os.path.exists(jobs_path):
    import pandas as pd
    num_jobs = len(pd.read_parquet(jobs_path))
    effective_jobs = num_jobs if VLM_LIMIT is None else min(VLM_LIMIT, num_jobs)
    print(f"VLM jobs to run: {effective_jobs:,}/{num_jobs:,}")
else:
    print(f"Warning: vlm_jobs.parquet not found at {jobs_path}")

cmd = ["python", "scripts/04_run_vlm_correction.py", "--config", "configs/colab_runtime.yaml"]
if VLM_LIMIT is not None:
    cmd += ["--limit", str(VLM_LIMIT)]

print(f"Running VLM correction with limit_arg='{limit_arg or 'FULL'}'")
print("Started at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
start = time.perf_counter()
result = subprocess.run(cmd)
elapsed = time.perf_counter() - start
print("Finished at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Stage 4 elapsed: {elapsed / 60:.2f} min ({elapsed:.1f} sec)")
if result.returncode != 0:
    raise RuntimeError(f"VLM correction failed with exit code {result.returncode}")

In [ ]:
# === QA Bước 4: VLM Correction ===
import pandas as pd

corrected_path = f"{OUTPUT_DIR}/vlm_corrected.parquet"
import os
if os.path.exists(corrected_path):
    corrected = pd.read_parquet(corrected_path)
    jobs = pd.read_parquet(f"{OUTPUT_DIR}/vlm_jobs.parquet") if os.path.exists(f"{OUTPUT_DIR}/vlm_jobs.parquet") else pd.DataFrame()
    print(f"Tổng dòng VLM output: {len(corrected):,}")
    print(f"Groups đã xử lý: {corrected[['frame_id', 'group_id']].drop_duplicates().shape[0]:,}/{len(jobs):,} jobs")
    print(f"Parse status: {corrected['parse_status'].value_counts().to_dict()}")
    if 'vlm_model' in corrected.columns:
        print(f"Model usage: {corrected['vlm_model'].value_counts(dropna=False).to_dict()}")
    
    changed = corrected[corrected['raw_text'] != corrected['corrected_text']]
    print(f"\nSố dòng text thay đổi: {len(changed)} ({len(changed)/max(1,len(corrected))*100:.1f}%)")
    if VLM_LIMIT is not None and len(jobs):
        print(f"\nLƯU Ý: Đang chạy smoke test VLM_LIMIT={VLM_LIMIT}. Kết quả merge/search sau đó chỉ là partial correction.")
    
    display(corrected.head(20))
    print("--- So sánh raw vs corrected ---")
    if len(changed):
        display(changed[['frame_id', 'group_id', 'line_idx', 'raw_text', 'corrected_text', 'vlm_model', 'parse_status']].head(20))
    else:
        print("Không có dòng thay đổi trong sample hiện tại.")
else:
    print("Chưa có vlm_corrected.parquet — hãy chạy Bước 4 trước.")

---
## Bước 5: Gộp kết quả (Merge)

Gộp dữ liệu OCR thô với dữ liệu đã được VLM sửa lỗi, tạo bảng tổng hợp cấp frame.

> 💡 Bước này KHÔNG cần GPU.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/05_merge_features.py --config configs/colab_runtime.yaml

In [ ]:
# === QA Bước 5a: Merged line-level ===
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_parquet(f"{OUTPUT_DIR}/ppocr_raw.parquet")
merged = pd.read_parquet(f"{OUTPUT_DIR}/ocr_merged_lines.parquet")
print(f"Tổng dòng merged: {len(merged):,}")
print(f"Dòng có VLM correction: {merged['vlm_corrected'].sum():,} ({merged['vlm_corrected'].mean()*100:.1f}%)")
print(f"Dòng text thay đổi: {merged['text_changed'].sum():,} ({merged['text_changed'].mean()*100:.1f}%)")
print(f"Số line raw có bị mất sau merge: {len(raw) - len(merged):,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
merged['vlm_corrected'].value_counts().plot(kind='bar', ax=axes[0], title='VLM corrected lines')
merged['text_changed'].value_counts().plot(kind='bar', ax=axes[1], title='Text changed lines')
plt.tight_layout()
plt.show()

display(merged[['frame_id', 'line_idx', 'group_id', 'region_type', 'ocr_text', 'corrected_text', 'normalized_text', 'vlm_corrected', 'text_changed', 'parse_status']].head(20))
print("--- Lines changed ---")
display(merged[merged['text_changed']][['frame_id', 'line_idx', 'group_id', 'ocr_text', 'corrected_text', 'risk_score', 'vlm_model']].head(30))

In [ ]:
# === QA Bước 5b: Frame-level summary ===
summary = pd.read_parquet(f"{OUTPUT_DIR}/ocr_frame_summary.parquet")
print(f"Tổng frames có OCR: {len(summary):,}")
print(f"Frames có VLM correction: {summary['vlm_corrected'].sum():,} ({summary['vlm_corrected'].mean()*100:.1f}%)")
print(f"Avg line_count/frame: {summary['line_count'].mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
summary['line_count'].hist(bins=25, ax=axes[0])
axes[0].set_title('Line count per frame')
summary['avg_confidence'].hist(bins=25, ax=axes[1])
axes[1].set_title('Avg confidence per frame')
plt.tight_layout()
plt.show()

display(summary[['frame_id', 'line_count', 'avg_confidence', 'vlm_corrected', 'vlm_model_ids', 'ocr_corrected_text']].head(10))

---
## Bước 6: Tạo tài liệu Elasticsearch (JSONL)

Chuyển đổi dữ liệu frame-level thành file JSONL sẵn sàng đẩy vào Elasticsearch.

> 💡 Bước này KHÔNG cần GPU.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/06_build_es_documents.py --config configs/colab_runtime.yaml

In [ ]:
# === Xem kết quả Bước 6 ===
import json

es_path = f"{OUTPUT_DIR}/es_documents.jsonl"
import os
if os.path.exists(es_path):
    with open(es_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"Tổng số documents: {len(lines)}")
    
    # Hiển thị document đầu tiên
    if lines:
        doc = json.loads(lines[0])
        print(f"\n--- Document mẫu ---")
        print(json.dumps(doc, ensure_ascii=False, indent=2)[:2000])
else:
    print("Chưa có es_documents.jsonl")

---
## Bước 6.5: Benchmark lựa chọn model VLM

Benchmark chạy trên cùng tập `vlm_jobs.parquet` để so sánh latency, parse success, text change rate và lỗi load/inference theo từng model.

> Lưu ý Colab: benchmark nhiều model liên tiếp dễ hết VRAM hoặc gặp model-specific adapter issue. Hãy bắt đầu với `BENCHMARK_SAMPLE_GROUPS=5..10`, sau đó tăng dần. Nếu một model lỗi, script vẫn ghi dòng `status=error` vào CSV để bạn không mất kết quả model khác.

In [ ]:
# Chạy benchmark 3 model trên sample nhỏ
%cd {PIPELINE_DIR}
models_arg = " ".join(BENCHMARK_MODELS)
benchmark_output = f"{OUTPUT_DIR}/model_benchmark.csv"
print(f"Models: {BENCHMARK_MODELS}")
print(f"Sample groups: {BENCHMARK_SAMPLE_GROUPS}")
!python scripts/run_vlm_benchmark.py --config configs/colab_runtime.yaml --models {models_arg} --sample_groups {BENCHMARK_SAMPLE_GROUPS} --output {benchmark_output}

In [ ]:
# === QA Benchmark model ===
import os
import pandas as pd
import matplotlib.pyplot as plt

benchmark_path = f"{OUTPUT_DIR}/model_benchmark.csv"
if os.path.exists(benchmark_path):
    bench = pd.read_csv(benchmark_path)
    display(bench)
    ok = bench[bench['status'].eq('ok')].copy()
    if len(ok):
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        ok.plot(kind='bar', x='model_id', y='avg_latency_sec_per_group', ax=axes[0], legend=False, title='Avg latency sec/group')
        ok.plot(kind='bar', x='model_id', y='parse_success_rate', ax=axes[1], legend=False, title='Parse success rate')
        ok.plot(kind='bar', x='model_id', y='text_change_rate', ax=axes[2], legend=False, title='Text change rate')
        for ax in axes:
            ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.show()
    errors = bench[bench['status'].ne('ok')]
    if len(errors):
        print("Một số model lỗi trong môi trường hiện tại. Đây là tín hiệu env/adapter, không nhất thiết là model kém.")
        display(errors[['model_id', 'status', 'error_message']])
else:
    print("Chưa có model_benchmark.csv")

---
## Tải kết quả về máy Local

Sau khi hoàn tất, bạn có thể download các file kết quả về máy cá nhân để chạy Bước 7 & 8 (Elasticsearch) ở local.

In [ ]:
# Nén toàn bộ kết quả thành file zip để download
import shutil
zip_path = shutil.make_archive(f'/content/pipeline_output_{VIDEO_ID}', 'zip', OUTPUT_DIR)
print(f"File zip: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")

# Download tự động (Colab)
from google.colab import files
files.download(zip_path)

---
## Tóm tắt các file output

| File | Mô tả |
|------|--------|
| `frame_registry.parquet` | Danh sách metadata toàn bộ frames |
| `ppocr_raw.parquet` | Kết quả OCR thô (line-level) |
| `ocr_risk.parquet` | Điểm rủi ro từng dòng OCR |
| `ocr_groups.parquet` | Kết quả gom nhóm + phân loại vùng |
| `vlm_jobs.parquet` | Danh sách công việc gửi cho VLM |
| `vlm_corrected.parquet` | Kết quả sửa lỗi từ VLM |
| `ocr_merged_lines.parquet` | Dữ liệu gộp raw + corrected (line-level) |
| `ocr_frame_summary.parquet` | Tóm tắt OCR cấp frame |
| `es_documents.jsonl` | Tài liệu sẵn sàng đẩy vào Elasticsearch |
| `model_benchmark.csv` | Benchmark latency/parse/change rate cho các model VLM |
| `crops/` | Ảnh crop gửi cho VLM |
| `vlm_cache.sqlite` | Cache kết quả VLM (tránh chạy lại) |

In [ ]:
# === Tổng kết kết quả toàn bộ pipeline ===
import os

print(f"=" * 60)
print(f"  TỔNG KẾT PIPELINE - {VIDEO_ID}")
print(f"=" * 60)

files_info = [
    ('frame_registry.parquet', 'Frame Registry'),
    ('ppocr_raw.parquet', 'PP-OCR Raw'),
    ('ocr_risk.parquet', 'Risk Scoring'),
    ('ocr_groups.parquet', 'OCR Groups'),
    ('vlm_jobs.parquet', 'VLM Jobs'),
    ('vlm_corrected.parquet', 'VLM Corrected'),
    ('ocr_merged_lines.parquet', 'Merged Lines'),
    ('ocr_frame_summary.parquet', 'Frame Summary'),
    ('es_documents.jsonl', 'ES Documents'),
    ('model_benchmark.csv', 'Model Benchmark'),
    ('vlm_cache.sqlite', 'VLM Cache'),
]

for fname, label in files_info:
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✅ {label:20s} | {fname:30s} | {size_kb:8.1f} KB")
    else:
        print(f"  ❌ {label:20s} | {fname:30s} | chưa tạo")

crops_dir = os.path.join(OUTPUT_DIR, 'crops')
if os.path.exists(crops_dir):
    n_crops = len([f for f in os.listdir(crops_dir) if f.endswith('.jpg')])
    print(f"  📁 Crops directory   | crops/                         | {n_crops} files")

print(f"=" * 60)